# Métricas de evaluación de modelos de lenguaje para aplicaciones empresariales

**Curso:** Procesamiento de Lenguaje Natural · Unidad 6 — Aplicaciones avanzadas y evaluación  
**Institución:** Universidad Tecnológica de Bolívar  
**Referencia:** Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3.ª ed.), Cap. 3 (perplejidad), Cap. 13 (BLEU) y Cap. 17 (evaluación de NER)  
**Compatibilidad con Python:** 3.10 · 3.11 · 3.12 · 3.13

## Contexto y Propósito

Este notebook reúne las **métricas más usadas para evaluar sistemas de PLN** y las aplica a cinco casos: F1 para un extractor de datos de facturas, BLEU y ROUGE para resúmenes de informes, perplejidad para filtrar textos poco fluidos y **BERTScore** para comparar respuestas de atención al cliente por significado y no solo por palabras. El quinto caso evalúa **salidas reales de dos modelos en español** (mT5 y GPT-2) para ver cómo las métricas reflejan que un modelo sin ajuste a la tarea no resume. BLEU y ROUGE se calculan también **a mano**, paso a paso, para entender qué miden. Todos los textos están en español. Termina con una guía para elegir la métrica según la tarea.

**Objetivo:** aprender a escoger, calcular e interpretar la métrica adecuada para cada tipo de sistema de PLN, y a no confiar en una sola cifra para decidir si un modelo está listo para producción.

## Ejemplos de Aplicación en el Mundo Real

Cada caso muestra, paso a paso, cómo se aplica la técnica de este notebook a un problema real.

### Gobierno y legal — Resúmenes de expedientes

- **Problema:** validar resúmenes de expedientes judiciales generados por un modelo.
- **Entrada:** expediente y resumen escrito por un abogado.
- **Cómo se aplica las métricas de generación:**
  1. ROUGE mide el solapamiento de palabras con el resumen del abogado.
  2. BERTScore mide la similitud de significado aunque cambien las palabras.
  3. Las fechas, montos y partes se verifican a mano, porque ninguna métrica comprueba hechos.
- **Salida:** puntajes ROUGE y BERTScore más una lista de errores factuales.
- **Decisión:** uso interno como borrador, siempre con revisión del abogado.

### Finanzas y servicio al cliente — Extractor de datos de facturas

- **Problema:** validar un extractor de NIT, fecha y total antes de automatizar la contabilidad.
- **Entrada:** facturas con los campos verificados a mano.
- **Cómo se aplica precisión, recall y F1:**
  1. Precisión = campos extraídos correctos / campos extraídos.
  2. Recall = campos extraídos correctos / campos que debían extraerse.
  3. F1 combina ambas por campo.
- **Salida:** F1 por campo (NIT, fecha, total).
- **Decisión:** automatizar solo los campos con F1 alto; el resto queda en revisión.

## Requisitos Previos

**Instalación** (una sola vez, desde la terminal; en Jupyter o Colab antepón `!`):

```bash
pip install evaluate sacrebleu rouge-score bert-score transformers torch pandas sentencepiece protobuf
```

**Librerías que se importan:**

| Librería | Para qué se usa |
|---------|-----------------|
| `math, re, collections` | Cálculo manual de BLEU, ROUGE y perplejidad |
| `torch` | Perplejidad y semilla del muestreo |
| `evaluate` | BLEU (`sacrebleu`) y ROUGE |
| `sacrebleu` | Su tokenizador `13a`, para que el BLEU manual coincida con el de la librería |
| `transformers` | GPT-2 en español para la perplejidad; mT5 para generar resúmenes (sección 6) |
| `bert_score` | BERTScore (similitud semántica con *embeddings* contextuales) |
| `pandas` | Guía de selección de métricas |

**Modelos** (se descargan de Hugging Face la primera vez):

| Modelo | Uso | Tamaño aprox. |
|---|---|---|
| [`PlanTL-GOB-ES/gpt2-base-bne`](https://huggingface.co/PlanTL-GOB-ES/gpt2-base-bne) | GPT-2 entrenado en español: perplejidad y generación | 0.5 GB |
| [`LeoCordoba/mt5-small-mlsum`](https://huggingface.co/LeoCordoba/mt5-small-mlsum) | mT5 ajustado para resumir en español | 1.2 GB |
| `bert-base-multilingual-cased` | Modelo que BERTScore usa por defecto con `lang='es'` | 0.7 GB |

## Marco metodológico: pipeline ML / LLM (4×3)

![Pipeline ML y LLM](img/Pipeline_ML_LLM_4x3.png)

El diagrama tiene una franja común de datos (arriba), dos rutas de modelado y una franja de operación (abajo): **ruta clásica** (*feature-based ML*) y **ruta LLM** (*foundation models*). Este notebook recorre la **ruta LLM / fundacional**. Etapas del diagrama que cubre:

| Etapa del diagrama | Dónde se aborda en este notebook |
|--------------------|----------------------------------|
| Test / Eval Set | Secciones 2–5: anotaciones de referencia (facturas), resúmenes y respuestas de referencia |
| Evaluation & QA | Secciones 2–7: F1, BLEU, ROUGE, perplejidad, BERTScore y guía de selección |
| Foundation / Pre-trained LLM | Sección 4: GPT-2 en español como medidor de fluidez; sección 5: BERT multilingüe dentro de BERTScore; sección 6: mT5 y GPT-2 como sistemas evaluados |

Este notebook se concentra en la caja **Evaluation & QA** del diagrama, común a la ruta clásica y a la ruta LLM: las mismas métricas sirven para evaluar un CRF, un BERT ajustado o un LLM con *prompting*.

## Estándares de referencia

Prácticas de este notebook alineadas con estándares de IEEE SA, ISO/IEC y la Open Source Initiative (OSI). Se citan como guía de buenas prácticas, no como certificación de cumplimiento.

| Estándar | Cómo se aplica aquí |
|----------|---------------------|
| **IEEE 3119-2025** — Adquisición (*procurement*) de IA y sistemas de decisión automatizada | Antes de comprar o adoptar un sistema de IA hay que definir con qué métricas y datos de referencia se evaluará; la guía de la sección 7 sirve de punto de partida. |
| **ISO/IEC 42001:2023** — Sistema de gestión de IA en una organización | Reportar varias métricas, con su significado y sus límites, es parte de la documentación que exige un sistema de gestión de IA. |
| **IEEE 7001-2021** — Transparencia de sistemas autónomos | Descomponer F1 en verdaderos positivos, falsos positivos y falsos negativos (sección 2.1) hace transparentes los tipos de error. |
| **ISO/IEC 23894:2023** — Guía para la gestión de riesgos de IA | Riesgo documentado: BLEU, ROUGE y BERTScore no verifican hechos; una respuesta con un dato falso puede puntuar alto. |
| **OSI · Open Source AI Definition 1.0 (2024)** — Qué hace que un sistema de IA sea de código abierto: libertades de usar, estudiar, modificar y compartir, con acceso a código, parámetros e información de los datos | Todas las métricas y modelos usados son abiertos y reproducibles. |

## Objetivos de aprendizaje

Al finalizar este notebook podrás:

1. Calcular **F1** para evaluar un sistema de NER en el procesamiento de documentos
2. Evaluar la calidad de textos generados con **BLEU** y **ROUGE**, y calcularlos a mano
3. Usar la **perplejidad** como métrica intrínseca de un modelo de lenguaje
4. Usar **BERTScore** para captar similitud semántica más allá del solapamiento de n-gramas
5. Evaluar salidas reales de **mT5** y **GPT-2** en español con BLEU, ROUGE y perplejidad, y explicar por qué un modelo no ajustado a la tarea puntúa bajo
6. Elegir la métrica adecuada para cada tarea de PLN con una guía de decisión

## ¿Por qué varias métricas?

Ninguna métrica capta todas las dimensiones de la calidad de un texto:

| Métrica | Qué capta | Qué no capta |
|---|---|---|
| F1 | Equilibrio entre precisión y *recall* | Orden, fluidez |
| BLEU | Precisión de n-gramas frente a la referencia | Significado, paráfrasis |
| ROUGE | Cobertura (*recall*) de n-gramas frente a la referencia | Fluidez, coherencia |
| Perplejidad | Confianza del modelo de lenguaje | Desempeño en la tarea |
| BERTScore | Similitud semántica | Exactitud de los hechos |

> ℹ️ Los datos de los casos (facturas, la noticia de *Aguas del Caribe*, respuestas de servicio) son **ejemplos inventados** para practicar; no describen hechos reales.

---

## 1. Configuración del entorno

In [ ]:
# Cálculo, métricas y modelos
import math
import re
import torch
import evaluate
from transformers import AutoTokenizer, AutoModelForCausalLM

---

## 2. Caso 1 — F1 para NER en el procesamiento de facturas

**Escenario:** un equipo de cuentas por pagar usa un sistema de NER para extraer las entidades `AMOUNT` (monto), `VENDOR` (proveedor) y `DATE` (fecha) de las facturas. Evaluamos su precisión y su *recall* en una muestra de 10 facturas.

*Simplificación:* aquí cada factura se representa por el **conjunto de tipos** de entidad encontrados; una evaluación completa compararía también el texto y la posición de cada entidad (como `seqeval`).

In [ ]:
# Entidades de referencia (anotadas a mano) en 10 facturas
gold_entities = [
    {'AMOUNT', 'VENDOR', 'DATE'},
    {'AMOUNT', 'DATE'},
    {'VENDOR', 'DATE'},
    {'AMOUNT', 'VENDOR'},
    {'AMOUNT', 'VENDOR', 'DATE'},
    {'DATE'},
    {'AMOUNT', 'VENDOR', 'DATE'},
    {'AMOUNT'},
    {'VENDOR', 'DATE'},
    {'AMOUNT', 'VENDOR', 'DATE'},
]

# Entidades que predijo el sistema de NER
pred_entities = [
    {'AMOUNT', 'VENDOR'},          # faltó DATE
    {'AMOUNT', 'DATE'},            # correcto
    {'VENDOR'},                    # faltó DATE
    {'AMOUNT', 'VENDOR', 'DATE'},  # DATE de más (falso positivo)
    {'AMOUNT', 'VENDOR', 'DATE'},  # correcto
    {'DATE'},                      # correcto
    {'AMOUNT', 'DATE'},            # faltó VENDOR
    {'AMOUNT'},                    # correcto
    {'VENDOR', 'DATE'},            # correcto
    {'AMOUNT', 'VENDOR', 'DATE'},  # correcto
]


def entity_f1(gold_list: list, pred_list: list) -> tuple:
    """Calcula precisión, recall y F1 micro sobre conjuntos de entidades.

    Args:
        gold_list: lista de conjuntos de etiquetas de referencia por documento.
        pred_list: lista de conjuntos de etiquetas predichas por documento.

    Returns:
        Tupla (precisión, recall, f1) con valores en [0, 1].
    """
    tp = sum(len(g & p) for g, p in zip(gold_list, pred_list))   # en ambos
    fp = sum(len(p - g) for g, p in zip(gold_list, pred_list))   # predichas de más
    fn = sum(len(g - p) for g, p in zip(gold_list, pred_list))   # no encontradas
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return precision, recall, f1


precision, recall, f1 = entity_f1(gold_entities, pred_entities)
print('Evaluación del NER en las facturas')
print(f'  Precisión: {precision:.4f}')
print(f'  Recall:    {recall:.4f}')
print(f'  F1:        {f1:.4f}')

> **Interpretación de la salida:** el F1 micro cuenta cada entidad individual en todos los documentos. Aquí la **precisión (0.95) es mayor que el *recall* (≈ 0.86)**: el sistema casi no inventa entidades (1 falso positivo), pero **se le escapan** algunas (3 falsos negativos). En un sistema de facturas el *recall* suele ser más crítico: no extraer un monto detiene el pago, mientras que una entidad de más se corrige en la revisión.

### 2.1 Cálculo manual de F1 — entender la fórmula

In [ ]:
# Conteo explícito de verdaderos positivos, falsos positivos y falsos negativos
tp_total = sum(len(g & p) for g, p in zip(gold_entities, pred_entities))
fp_total = sum(len(p - g) for g, p in zip(gold_entities, pred_entities))
fn_total = sum(len(g - p) for g, p in zip(gold_entities, pred_entities))

print(f'Verdaderos positivos (VP): {tp_total}  — entidades bien predichas')
print(f'Falsos positivos     (FP): {fp_total}  — predichas pero no estaban en la referencia')
print(f'Falsos negativos     (FN): {fn_total}  — estaban en la referencia pero no se predijeron')
print()
print(f'Precisión = VP / (VP + FP) = {tp_total} / {tp_total + fp_total} = {tp_total / (tp_total + fp_total):.4f}')
print(f'Recall    = VP / (VP + FN) = {tp_total} / {tp_total + fn_total} = {tp_total / (tp_total + fn_total):.4f}')

> **Interpretación de la salida:** separar los aciertos y errores hace transparentes los tipos de error. Muchos FP indican que el modelo **sobrepredice** (detecta entidades que no existen); muchos FN, que **subpredice** (se le escapan). Si el modelo da un puntaje de confianza, subir el umbral reduce los FP y bajarlo reduce los FN; más datos de entrenamiento suelen ayudar con ambos.

---

## 3. Caso 2 — BLEU y ROUGE para resumir una noticia

**Escenario:** dos sistemas producen resúmenes de una noticia (ficticia) sobre una inversión en el acueducto de Cartagena. Usamos BLEU y ROUGE para compararlos con un resumen escrito por una persona. Los dos resúmenes se escribieron a mano para ilustrar dos estilos: uno que **reutiliza las palabras** de la referencia pero **cambia una cifra**, y otro que **parafrasea** sin errores.

In [ ]:
bleu = evaluate.load('sacrebleu')
rouge = evaluate.load('rouge')


def tokenizar_es(texto: str) -> list:
    """Tokenizador para ROUGE que conserva tildes y ñ (el de rouge-score solo conserva a–z y 0–9)."""
    return re.findall(r'\w+', texto.lower())


# Resumen de referencia (ficticio) escrito por una persona
reference = ('Aguas del Caribe ampliará el acueducto de Cartagena antes de diciembre '
             'con una inversión de 80.000 millones de pesos.')

# Resúmenes de ejemplo de dos sistemas (escritos a mano para ilustrar dos estilos)
system_summaries = {
    'Sistema A (reutiliza palabras, cifra falsa)': (
        'Aguas del Caribe ampliará el acueducto de Cartagena antes de diciembre '
        'con una inversión de 8.000 millones de pesos.'
    ),
    'Sistema B (parafrasea, correcto)': (
        'La ampliación del acueducto cartagenero costará 80.000 millones de pesos '
        'y estará lista a fin de año.'
    ),
}

for model_name, summary in system_summaries.items():
    b = bleu.compute(predictions=[summary], references=[[reference]])
    r = rouge.compute(predictions=[summary], references=[reference], tokenizer=tokenizar_es)
    print(f'[{model_name}]')
    print(f'  Resumen: {summary}')
    print(f'  BLEU:    {b["score"]:.2f}')
    print(f'  ROUGE-1: {r["rouge1"]:.4f}')
    print(f'  ROUGE-L: {r["rougeL"]:.4f}')
    print()

> **Interpretación de la salida:** el sistema A obtiene puntajes casi perfectos (BLEU ≈ 85.8, ROUGE-L ≈ 0.95) porque copia la redacción de la referencia, aunque **la cifra está mal** (8.000 en lugar de 80.000 millones): un error grave para cualquier lector. El sistema B dice lo correcto con otras palabras y obtiene BLEU ≈ 14.3 y ROUGE-L ≈ 0.37. Este es el límite central de BLEU y ROUGE: miden **coincidencia de palabras**, no significado ni verdad.

> **Tokenización en español:** el tokenizador por defecto de `rouge-score` elimina todo lo que no sea *a–z* o *0–9*: *ampliará* se partiría en *ampliar* y *á*, y *año* en *a* y *o*. Por eso se le pasa `tokenizar_es`.

### 3.1 Cálculo manual de BLEU y ROUGE — entender las fórmulas

**BLEU** combina la precisión de n-gramas (de 1 a 4 palabras) con una penalización por brevedad (*BP*):

$$\text{BLEU} = BP \cdot \exp\left(\frac{1}{4}\sum_{n=1}^{4} \log p_n\right), \qquad BP = \begin{cases} 1 & \text{si } c > r \\ e^{\,1 - r/c} & \text{si } c \le r \end{cases}$$

donde $p_n$ = n-gramas del candidato que aparecen en la referencia (cada uno contado como máximo las veces que aparece en ella) / n-gramas del candidato, $c$ = longitud del candidato y $r$ = longitud de la referencia.

**ROUGE-1** cuenta las palabras en común; **ROUGE-L** usa la subsecuencia común más larga (LCS), que respeta el orden pero admite huecos. Ambas se reportan como F1 entre precisión (sobre el candidato) y *recall* (sobre la referencia).

In [ ]:
import math, re
from collections import Counter
from sacrebleu.tokenizers.tokenizer_13a import Tokenizer13a

tok13a = Tokenizer13a()   # el mismo tokenizador que usa sacrebleu por defecto


def ngramas(tokens: list, n: int) -> Counter:
    """Cuenta los n-gramas de una lista de tokens."""
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))


def bleu_manual(candidato: str, referencia: str, max_n: int = 4) -> float:
    """BLEU de una oración con una referencia, paso a paso (0–100)."""
    c, r = tok13a(candidato).split(), tok13a(referencia).split()
    log_p = 0.0
    for n in range(1, max_n + 1):
        cn, rn = ngramas(c, n), ngramas(r, n)
        coinciden = sum(min(k, rn[g]) for g, k in cn.items())   # conteo recortado
        total = max(sum(cn.values()), 1)
        print(f'    p{n} = {coinciden}/{total} = {coinciden / total:.4f}')
        log_p += math.log(coinciden / total) / max_n            # supone coinciden > 0
    bp = 1.0 if len(c) > len(r) else math.exp(1 - len(r) / len(c))
    print(f'    BP = {bp:.4f}  (longitud candidato {len(c)}, referencia {len(r)})')
    return 100 * bp * math.exp(log_p)


def lcs(a: list, b: list) -> int:
    """Longitud de la subsecuencia común más larga (programación dinámica)."""
    t = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i, x in enumerate(a):
        for j, y in enumerate(b):
            t[i + 1][j + 1] = t[i][j] + 1 if x == y else max(t[i][j + 1], t[i + 1][j])
    return t[-1][-1]


def f1(p: float, r: float) -> float:
    return 2 * p * r / (p + r) if p + r else 0.0


def rouge_manual(candidato: str, referencia: str) -> tuple:
    """ROUGE-1 y ROUGE-L (F1) con el mismo tokenizador que tokenizar_es."""
    c, r = tokenizar_es(candidato), tokenizar_es(referencia)
    comunes = sum((Counter(c) & Counter(r)).values())
    l = lcs(c, r)
    print(f'    ROUGE-1: {comunes} palabras en común → P = {comunes}/{len(c)}, R = {comunes}/{len(r)}')
    print(f'    ROUGE-L: subsecuencia común más larga = {l} → P = {l}/{len(c)}, R = {l}/{len(r)}')
    return f1(comunes / len(c), comunes / len(r)), f1(l / len(c), l / len(r))


for nombre, resumen in system_summaries.items():
    print(f'[{nombre}]')
    b_man = bleu_manual(resumen, reference)
    r1_man, rl_man = rouge_manual(resumen, reference)
    b_lib = bleu.compute(predictions=[resumen], references=[[reference]])['score']
    r_lib = rouge.compute(predictions=[resumen], references=[reference], tokenizer=tokenizar_es)
    print(f'    BLEU     manual = {b_man:.2f} | librería = {b_lib:.2f}')
    print(f'    ROUGE-1  manual = {r1_man:.4f} | librería = {r_lib["rouge1"]:.4f}')
    print(f'    ROUGE-L  manual = {rl_man:.4f} | librería = {r_lib["rougeL"]:.4f}\n')

> **Interpretación de la salida:** el cálculo manual coincide con la librería. En A, 19 de 20 unigramas coinciden: solo falla *8.000*, y por eso los cuatro $p_n$ son altos. En B solo 8 de 18 palabras coinciden y casi ningún 4-grama, así que el promedio geométrico se desploma; además B es más corto que la referencia y recibe una penalización por brevedad (BP ≈ 0.89). Ver las fracciones muestra por qué una paráfrasis correcta puntúa bajo: BLEU premia repetir **secuencias exactas** de la referencia.

---

## 4. Caso 3 — Perplejidad para evaluar la calidad del texto

**Escenario:** un equipo de contenidos quiere detectar automáticamente si un texto generado es lo bastante fluido para publicarse. La perplejidad da un puntaje automático de fluidez.

**Fórmula:** $\text{PPL}(texto) = \exp\left(-\frac{1}{N} \sum_i \log P(token_i \mid contexto)\right)$

Menor perplejidad = el modelo asigna más probabilidad al texto = texto más fluido o esperado.

In [ ]:
# GPT-2 entrenado en español, en modo evaluación
PPL_MODEL = 'PlanTL-GOB-ES/gpt2-base-bne'
gpt2_tokenizer = AutoTokenizer.from_pretrained(PPL_MODEL)
gpt2_model = AutoModelForCausalLM.from_pretrained(PPL_MODEL)
gpt2_model.eval()


def compute_perplexity(model, tokenizer, text: str) -> float:
    """Calcula la perplejidad de un modelo de lenguaje causal para un texto.

    Args:
        model: modelo causal (GPT-2) en modo evaluación.
        tokenizer: su tokenizador.
        text: texto a evaluar.

    Returns:
        Perplejidad (float; menor es mejor).
    """
    encodings = tokenizer(text, return_tensors='pt')
    with torch.no_grad():
        outputs = model(**encodings, labels=encodings['input_ids'])
    return math.exp(outputs.loss.item())


# Tres textos: fluido, con el orden alterado y en otro idioma
texts = {
    'Fluido (humano)': 'El informe trimestral muestra un fuerte crecimiento de los ingresos en todos los segmentos.',
    'Poco fluido (desordenado)': 'trimestral informe muestra ingresos fuerte crecer todos segmentos los en.',
    'Inglés en un modelo de español': 'The quarterly report shows strong revenue growth across all segments.',
}

for label, text in texts.items():
    ppl = compute_perplexity(gpt2_model, gpt2_tokenizer, text)
    print(f'[{label}]')
    print(f'  Texto: {text}')
    print(f'  Perplejidad: {ppl:.2f}')
    print()

> **Interpretación de la salida:** se espera que el texto fluido obtenga la perplejidad más baja (el modelo asigna alta probabilidad al español natural), el texto desordenado una más alta y el texto en inglés la más alta, porque este GPT-2 se entrenó con textos en español. Por eso la perplejidad sirve como filtro de fluidez y para detectar textos en otro idioma, pero **no** dice si el contenido es correcto.

---

## 5. Caso 4 — BERTScore: evaluación semántica

**Escenario:** dos sistemas de atención al cliente responden la misma pregunta con distinta redacción. BLEU castiga la paráfrasis; BERTScore mide la similitud semántica usando *embeddings* contextuales.

In [ ]:
from bert_score import score as bert_score_fn

# Respuesta de referencia y dos respuestas de sistema (ficticias)
reference_response = 'El saldo de su cuenta es de 500.000 pesos. Por favor acérquese a la sucursal para actualizar sus datos.'

system_responses = [
    'Su cuenta tiene un saldo actual de 500.000 COP. Para actualizar su información, visite nuestra oficina.',  # paráfrasis correcta
    'Lo siento, no encuentro su cuenta. Intente de nuevo más tarde.',  # respuesta incorrecta
]

references = [reference_response, reference_response]

# lang='es' usa BERT multilingüe; rescale_with_baseline=True reescala los puntajes para que se distingan mejor
P, R, F1 = bert_score_fn(system_responses, references, lang='es', rescale_with_baseline=True, verbose=False)
print('Resultados de BERTScore (precisión / recall / F1):')
labels = ['Paráfrasis (correcta)', 'Respuesta incorrecta']
for label, p, r, f in zip(labels, P, R, F1):
    print(f'  [{label}] P={p:.4f}  R={r:.4f}  F1={f:.4f}')

> **Interpretación de la salida:** BERTScore compara cada token de la respuesta con los de la referencia usando *embeddings* contextuales (con `lang='es'`, de `bert-base-multilingual-cased`), así que reconoce paráfrasis que BLEU no ve. Se espera que la paráfrasis correcta obtenga un F1 claramente mayor que la respuesta incorrecta. Sin `rescale_with_baseline`, los puntajes brutos suelen quedar todos entre 0.8 y 0.95 y la diferencia parece pequeña. Ojo: BERTScore tampoco verifica hechos; una respuesta con un saldo equivocado pero redactada igual puntuaría alto.

---

## 6. Caso 5 — Evaluar salidas reales de dos modelos (mT5 y GPT-2)

**Escenario:** hasta aquí los textos evaluados se escribieron a mano. Ahora se generan con dos modelos reales en español y se evalúan con las mismas métricas: **mT5** (codificador–decodificador, ajustado para resumir noticias) y **GPT-2** (solo decodificador, modelo de lenguaje general).

> Esta sección integra el antiguo notebook `classification_evaluation.ipynb`.

In [ ]:
from transformers import AutoModelForSeq2SeqLM

torch.manual_seed(42)   # GPT-2 genera con muestreo: la semilla hace reproducible el ejemplo

# mT5 ajustado para resumir en español (MLSUM)
T5_MODEL = 'LeoCordoba/mt5-small-mlsum'
t5_tokenizer = AutoTokenizer.from_pretrained(T5_MODEL)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(T5_MODEL)

# Texto a resumir (noticia ficticia) y resumen de referencia (escrito por una persona)
input_text = ('La alcaldesa Marta Ruiz anunció que Aguas del Caribe ampliará el acueducto de Cartagena '
              'antes de diciembre, con una inversión de 80.000 millones de pesos.')
reference_summary = 'Aguas del Caribe ampliará el acueducto de Cartagena antes de diciembre.'

# mT5: recibe el texto sin prefijo; decodificación con búsqueda en haz (determinista)
ids_t5 = t5_model.generate(**t5_tokenizer(input_text, return_tensors='pt'), max_length=40, num_beams=4)
text_t5 = t5_tokenizer.decode(ids_t5[0], skip_special_tokens=True)

# GPT-2 (cargado en la sección 4): no sabe resumir, continúa el texto
inputs_gpt2 = gpt2_tokenizer(input_text, return_tensors='pt')
ids_gpt2 = gpt2_model.generate(
    **inputs_gpt2,
    max_new_tokens=30, do_sample=True, temperature=0.7, top_k=50,
    pad_token_id=gpt2_tokenizer.eos_token_id,   # GPT-2 no tiene token de relleno
)
text_gpt2 = gpt2_tokenizer.decode(ids_gpt2[0][inputs_gpt2['input_ids'].shape[1]:], skip_special_tokens=True)

# Mismas métricas para ambas salidas
print(f'Referencia: {reference_summary}\n')
for name, out in [('mT5', text_t5), ('GPT-2', text_gpt2)]:
    b = bleu.compute(predictions=[out], references=[[reference_summary]])
    r = rouge.compute(predictions=[out], references=[reference_summary], tokenizer=tokenizar_es)
    # La perplejidad necesita al menos un token
    ppl = compute_perplexity(gpt2_model, gpt2_tokenizer, out) if out.strip() else float('nan')
    print(f'[{name}] {out}')
    print(f'  BLEU={b["score"]:.2f}  ROUGE-1={r["rouge1"]:.4f}  ROUGE-L={r["rougeL"]:.4f}  Perplejidad (GPT-2)={ppl:.2f}\n')

> **Interpretación de la salida:** mT5 produce un resumen corto, a menudo con estilo de titular, porque se ajustó con resúmenes de noticias; GPT-2, sin ajuste para la tarea, **continúa** el texto en lugar de resumirlo. BLEU y ROUGE lo reflejan con puntajes más bajos para GPT-2. La perplejidad, en cambio, puede ser **baja para GPT-2** porque se mide con el mismo GPT-2 que generó el texto: un modelo encuentra “natural” lo que él mismo escribe. Por eso la perplejidad no sirve para comparar sistemas evaluados con el mismo modelo que los generó, y las métricas con referencia miden la tarea, no la fluidez.

---

## 7. Guía para elegir la métrica

In [ ]:
import pandas as pd

# Métrica principal recomendada por tipo de tarea
guide = pd.DataFrame([
    {'Tarea': 'Reconocimiento de entidades (NER)', 'Métrica principal': 'F1 por entidad (micro/macro)', 'Por qué': 'Equilibra precisión y recall en etiquetas de spans'},
    {'Tarea': 'Traducción automática', 'Métrica principal': 'BLEU (y chrF)', 'Por qué': 'Métrica estándar de los benchmarks de traducción'},
    {'Tarea': 'Resumen de textos', 'Métrica principal': 'ROUGE-1 / ROUGE-L', 'Por qué': 'Mide la cobertura del contenido frente a la referencia'},
    {'Tarea': 'Filtro de fluidez', 'Métrica principal': 'Perplejidad', 'Por qué': 'Medida intrínseca de qué tan natural es el texto'},
    {'Tarea': 'Respuestas de atención al cliente', 'Métrica principal': 'BERTScore F1 + revisión humana', 'Por qué': 'Capta equivalencia semántica y admite paráfrasis'},
    {'Tarea': 'Clasificación de documentos', 'Métrica principal': 'Accuracy + F1 macro', 'Por qué': 'Accuracy si las clases están balanceadas; F1 macro si no'},
])
print(guide.to_string(index=False))

> **Interpretación de la salida:** la guía resume cuándo usar cada métrica. En la práctica conviene reportar **varias métricas** para tener una visión completa. En un proyecto nuevo, empieza con la métrica principal de la tabla y añade BERTScore y evaluación humana para las salidas más críticas.

---

## Resumen

| Métrica | Librería | Cuándo usarla |
|---|---|---|
| F1 | `sklearn.metrics`, `seqeval` | Clasificación, NER, recuperación de información |
| BLEU | `evaluate` (sacrebleu); a mano en la sección 3.1 | Traducción y seq2seq con referencia fija |
| ROUGE | `evaluate` (rouge-score) con un tokenizador para español; a mano en la sección 3.1 | Resumen y generación abstractiva |
| Perplejidad | `transformers` | Calidad del modelo de lenguaje, filtro de fluidez |
| BERTScore | `bert-score` | Cualquier tarea donde la paráfrasis sea aceptable |

> Las métricas de las secciones 2–6 sirven igual para evaluar los modelos de `text_generation_business.ipynb`.